In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, to_hetero
from torch_geometric.data import DataLoader
from halide_gnn_cost_model.data import PipelineDataset
from pathlib import Path

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
dataset = PipelineDataset(Path("../resources/pipelines"))
data = dataset[0]  # Get the first pipeline graph
print(data.metadata)

<bound method HeteroData.metadata of HeteroData(
  y=[5],
  function={ x=[4, 1] },
  (function, called_by, function)={ edge_index=[2, 4] }
)>


In [3]:
data

HeteroData(
  y=[5],
  function={ x=[4, 1] },
  (function, called_by, function)={ edge_index=[2, 4] }
)

In [4]:
class PipeGCN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers=2):
        super(PipeGCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(-1, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(-1, hidden_channels))
        self.convs.append(GCNConv(-1, out_channels))

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [5]:
gcn = PipeGCN(hidden_channels=32, out_channels=32, num_layers=3)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()


In [6]:
out = gcn(data.x_dict, data.edge_index_dict)
out["function"][0]

tensor([-0.2488, -0.0903, -0.1424, -0.4248,  0.2222, -0.2895, -0.3943, -0.0566,
         0.3531,  0.2539,  0.2022, -0.1482,  0.1110,  0.2849, -0.4268, -0.4426,
        -0.1588, -0.0916,  0.0194, -0.2057,  0.1304,  0.1447,  0.1118, -0.1718,
        -0.3477,  0.2528,  0.2268,  0.2486,  0.3196, -0.2820,  0.0688,  0.2627],
       grad_fn=<SelectBackward0>)

In [7]:
class PipelineModel(torch.nn.Module):
    def __init__(self, gnn, out_channels, num_runtime):
        super(PipelineModel, self).__init__()
        self.function_gnn = gnn
        self.pipeline_lin = torch.nn.Linear(out_channels, num_runtime)

    def forward(self, data, ptr=None):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        out = self.function_gnn(x_dict, edge_index_dict)
        # Get the feature of the pipeline node
        idx = 0 if ptr is None else ptr
        pipeline_feat = out["function"][idx]
        # Predict the runtime
        x = self.pipeline_lin(pipeline_feat)
        run_time = torch.exp(x)
        return run_time

In [8]:
model = PipelineModel(gcn, 32, 5)
model

PipelineModel(
  (function_gnn): GraphModule(
    (convs): ModuleList(
      (0-2): 3 x ModuleDict(
        (function__called_by__function): GCNConv(-1, 32)
      )
    )
  )
  (pipeline_lin): Linear(in_features=32, out_features=5, bias=True)
)

In [9]:
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)
data_loader

/var/folders/qh/c7l883gn5s548l5w18l25cwc0000gn/T/nix-shell.a4AiSO/ipykernel_22566/1643914774.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  data_loader = DataLoader(dataset, batch_size=4, shuffle=True)


In [10]:
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.L1Loss()

for epoch in range(100):
    model.train()
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        pred = model(batch, batch["function"].ptr[:-1])
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(data_loader)}")

Epoch 10, Loss: 823.0026286315918
Epoch 20, Loss: 765.5187671661376
Epoch 30, Loss: 758.8854083824158
Epoch 40, Loss: 757.3038373565673
Epoch 50, Loss: 754.9516981506348
Epoch 60, Loss: 757.4931498336792
Epoch 70, Loss: 759.9795962524414
Epoch 80, Loss: 755.3832987785339
Epoch 90, Loss: 756.6462959671021
Epoch 100, Loss: 754.2211414527893


In [11]:
torch.set_printoptions(precision=4)
print(model(data))

tensor([  1.5572,   7.8874,  35.3432, 146.8430, 810.9636],
       grad_fn=<ExpBackward0>)
